

 **Trajectory Segmentation & Clustering:**
   - **Phase One (Partition):** Segments each trajectory using a simplified MDL optimization algorithm (with complexity O(n)).
   - **Phase Two (Group):** Clusters the segments with a DBSCAN-like algorithm using a custom distance measure that combines parallel, perpendicular, and angular differences. 
   - For each cluster, a representative trajectory is computed.




In [1]:
import h5py
import numpy as np
import plotly.graph_objs as go
from scipy.stats import entropy
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt
 

# For reproducibility
np.random.seed(42)

## Entropy Calculation Functions

Jamies code, wanted to have here so I could cluster after we do L0 separation

In [ ]:
def load_data_from_hdf5(file_path, demo_name, data_type='states', obs_type='obs'):
    with h5py.File(file_path, 'r') as f:
        data_path = f'data/{demo_name}/{obs_type}/{data_type}'
        if data_path not in f:
            raise KeyError(f"Key '{data_type}' not found at {data_path}.")
        states = f[data_path][:]
        return states


def extract_translation_data(states):
    # Assumes the first 3 columns are X, Y, Z translations
    return states[:, :3]


def compute_entropy(data, bins=30):
    # Calculate the histogram of the data in 3D space
    hist, edges = np.histogramdd(data, bins=bins, density=True)

    # Flatten the histogram and filter out zero probabilities
    prob_dist = hist.flatten()
    prob_dist = prob_dist[prob_dist > 0]  

    # Compute entropy using the probability distribution
    entropy_value = entropy(prob_dist)

    # Normalize entropy between 0 and 1
    max_entropy = np.log(bins ** 3)  
    normalized_entropy = entropy_value / max_entropy

    # Non-linear (exponential) scaling for better separation
    scaled_entropy = np.exp(normalized_entropy) - 1
    scaled_entropy = np.clip(scaled_entropy, 0, 1)

    return scaled_entropy


def compute_diff_entropy(demo_translations, bins=30):
    # Compute the difference between consecutive translations
    differences = np.diff(demo_translations, axis=0)
    return compute_entropy(differences, bins)


def classify_dataset(std_entropy, threshold=0.10):
    if std_entropy <= threshold:
        return "Expert"
    else:
        return "Play"


def create_3d_overlay_plot(all_translations, all_demo_names, entropy_values, overall_entropy, std_entropy, classification):
    traces = []
    for i, translations in enumerate(all_translations):
        x, y, z = translations[:, 0], translations[:, 1], translations[:, 2]
        trace = go.Scatter3d(
            x=x, y=y, z=z,
            mode='lines',
            name=f'Demo {all_demo_names[i]} (Entropy: {entropy_values[i]:.2f})',
            line=dict(width=3)
        )
        traces.append(trace)
    
    entropy_text = f'Overall Entropy: {overall_entropy:.2f}'
    std_diff_text = f'Standard Deviation: {std_entropy:.2f}'
    classification_text = f'Classification: {classification}'
    
    annotation = dict(
        text=f"{entropy_text}<br>{std_diff_text}<br>{classification_text}",
        showarrow=False,
        x=1, y=1,
        xref='paper', yref='paper',
        font=dict(size=14, color='red')
    )
    layout = go.Layout(
        title="Translation Data for Demos",
        scene=dict(xaxis=dict(title='X'), yaxis=dict(title='Y'), zaxis=dict(title='Z')),
        annotations=[annotation]
    )
    fig = go.Figure(data=traces, layout=layout)
    fig.show()


# Example usage for entropy calculation (adjust the file path and demo names as needed):
def entropy_pipeline(hdf5_file_path):
    with h5py.File(hdf5_file_path, 'r') as f:
        demos = list(f['data'].keys())
        all_translations, all_demo_names, entropy_values = [], [], []
        
        for demo_name in demos:
            try:
                states = load_data_from_hdf5(hdf5_file_path, demo_name)
                translations = extract_translation_data(states)
                all_translations.append(translations)
                all_demo_names.append(demo_name)
            except KeyError as e:
                print(f"Skipping {demo_name}: {e}")
        
        # Compute entropy for each demo
        for translations in all_translations:
            entropy_values.append(compute_diff_entropy(translations))

        overall_entropy = np.mean(entropy_values)
        std_entropy = np.std(entropy_values)
        classification = classify_dataset(std_entropy)
        
        create_3d_overlay_plot(all_translations, all_demo_names, entropy_values, overall_entropy, std_entropy, classification)
        
        return all_translations  # return trajectories for the next phase



## Trajectory Segmentation 

Below is a simplified implementation of an MDL optimization–style segmentation. The idea is to approximate a trajectory with a line segment and recursively segment if the approximation error exceeds a threshold.

In [ ]:
def mdl_segmentation(traj, threshold=0.05):
    """
    Recursively segment a trajectory (array of shape (N, 3))
    based on the maximum perpendicular distance from a line
    connecting the start and end of the segment.
    Returns a list of segments (each segment is a 2x3 array: [start, end]).
    """
    if traj.shape[0] < 2:
        return []
    
    start, end = traj[0], traj[-1]
    line_vec = end - start
    line_len = np.linalg.norm(line_vec)
    if line_len == 0:
        return [np.array([start, end])]
    
    # Compute distance from each point to the line
    t = np.dot(traj - start, line_vec) / (line_len**2)
    proj = start + np.outer(t, line_vec)
    dists = np.linalg.norm(traj - proj, axis=1)
    
    # Find the index of the maximum distance
    idx = np.argmax(dists)
    max_dist = dists[idx]
    
    # If the maximum distance is greater than the threshold, split and recurse
    if max_dist > threshold and idx not in [0, traj.shape[0]-1]:
        seg1 = mdl_segmentation(traj[:idx+1], threshold)
        seg2 = mdl_segmentation(traj[idx:], threshold)
        return seg1 + seg2
    else:
        return [np.array([start, end])]


def segment_all_trajectories(trajectories, threshold=0.05):
    """
    Apply segmentation to a list of trajectories.
    Returns a list of segments across all trajectories.
    """
    all_segments = []
    for traj in trajectories:
        segments = mdl_segmentation(traj, threshold)
        all_segments.extend(segments)
    return all_segments


## Custom Segment Distance & Clustering 

We define a custom distance between segments based on three components:

- **Parallel distance:** The difference along the main direction of one segment.
- **Perpendicular distance:** The orthogonal distance between the midpoints of the segments.
- **Angular distance:** The difference in orientation between the segments.

Then we use DBSCAN (with this custom metric) to cluster these segments.

In [4]:
def segment_to_vector(segment):
    """
    Convert a segment (2x3 array) to a 6D vector [start_x, start_y, start_z, end_x, end_y, end_z]
    """
    return segment.flatten()


def segment_distance(x, y, w_parallel=1.0, w_perp=1.0, w_angle=1.0):
    """
    Custom distance between two segments (given as 6D vectors).
    """
    seg1 = x.reshape(2, 3)
    seg2 = y.reshape(2, 3)
    
    # Compute midpoints
    mid1 = (seg1[0] + seg1[1]) / 2
    mid2 = (seg2[0] + seg2[1]) / 2
    
    # Compute direction vectors
    v1 = seg1[1] - seg1[0]
    v2 = seg2[1] - seg2[0]
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    if norm1 == 0 or norm2 == 0:
        return np.linalg.norm(mid1 - mid2)
    d1 = v1 / norm1
    d2 = v2 / norm2
    
    # Parallel distance: projection of (mid2 - mid1) on d1
    proj = np.dot(mid2 - mid1, d1)
    parallel_dist = np.abs(proj)
    
    # Perpendicular distance: norm of the component orthogonal to d1
    perp_vec = mid2 - mid1 - proj * d1
    perp_dist = np.linalg.norm(perp_vec)
    
    # Angular distance: angle between d1 and d2
    dot = np.clip(np.dot(d1, d2), -1.0, 1.0)
    angle = np.arccos(dot)
    
    total_distance = w_parallel * parallel_dist + w_perp * perp_dist + w_angle * angle
    return total_distance


def cluster_segments(segments, eps=0.5, min_samples=3):
    """
    Cluster segments using DBSCAN with a custom metric.
    Each segment is converted to a 6D vector.
    """
    vectors = np.array([segment_to_vector(seg) for seg in segments])
    
    db = DBSCAN(eps=eps, min_samples=min_samples, metric=segment_distance)
    labels = db.fit_predict(vectors)
    return labels, vectors

def representative_trajectory(cluster_segments):
    """
    Given a list of segments (each as a 2x3 array) belonging to a cluster,
    compute a representative trajectory as the average of the start and end points.
    """
    seg_vectors = np.array([segment_to_vector(seg) for seg in cluster_segments])
    # Average start points and end points separately
    starts = seg_vectors[:, :3]
    ends = seg_vectors[:, 3:]
    rep_start = np.mean(starts, axis=0)
    rep_end = np.mean(ends, axis=0)
    return np.vstack([rep_start, rep_end])

# Example: clustering dummy segments
# dummy_segments = [np.array([[0, 0, 0], [1, 1, 1]]), np.array([[0.1, 0, 0], [1.1, 1, 1]]), ... ]


## Putting It All Together



1. Loads the trajectories from an HDF5 file and computes the entropy separation and classification.
2. Segments each trajectory using the MDL segmentation function.
3. Clusters all segments using DBSCAN with the custom distance measure.
4. For each cluster, computes a representative trajectory.



In [ ]:
def full_pipeline(hdf5_file_path):
    # --- Entropy Separation Phase ---
    print("Running entropy separation...")
    trajectories = entropy_pipeline(hdf5_file_path)  
    
    # --- Segmentation Phase ---
    print("Segmenting trajectories...")
    segments = segment_all_trajectories(trajectories, threshold=0.05)
    print(f"Total segments obtained: {len(segments)}")
    
    # --- Clustering Phase ---
    print("Clustering segments...")
    labels, seg_vectors = cluster_segments(segments, eps=0.5, min_samples=3)
    
    # Organize segments by cluster label
    cluster_dict = {}
    for label, seg in zip(labels, segments):
        if label == -1:  # Noise
            continue
        cluster_dict.setdefault(label, []).append(seg)
    
    print(f"Found {len(cluster_dict)} clusters.")
    
    # Compute representative trajectory for each cluster
    rep_trajectories = {}
    for label, segs in cluster_dict.items():
        rep_traj = representative_trajectory(segs)
        rep_trajectories[label] = rep_traj
        print(f"Cluster {label}: Representative trajectory from {rep_traj[0]} to {rep_traj[1]}")
    

    fig = go.Figure()
    
    # Plot all segments colored by cluster
    colors = ['red', 'green', 'blue', 'orange', 'purple', 'cyan', 'magenta']
    for label, segs in cluster_dict.items():
        for seg in segs:
            fig.add_trace(go.Scatter3d(
                x=seg[:, 0], y=seg[:, 1], z=seg[:, 2],
                mode='lines',
                line=dict(color=colors[label % len(colors)], width=2),
                showlegend=False
            ))
    
    # Plot representative trajectories
    for label, rep in rep_trajectories.items():
        fig.add_trace(go.Scatter3d(
            x=rep[:, 0], y=rep[:, 1], z=rep[:, 2],
            mode='lines+markers',
            line=dict(color='black', width=4),
            name=f'Cluster {label} Rep'
        ))
    
    fig.update_layout(
        title='Clustered Trajectory Segments',
        scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z')
    )
    fig.show()

#  set your HDF5 file path
full_pipeline('../../expert_demos.hdf5')

Running entropy separation...


Segmenting trajectories...
Total segments obtained: 6
Clustering segments...
Found 0 clusters.
